# Improvement — DistilBERT v2 (full data + class weight + more epochs)

**Target:** menembus 80%+ dengan menyerang bottleneck NEUTRAL (v1: F1 0.40).

**Perubahan vs v1:**
- Data: full 200k train (bukan subset 30k)
- Epoch: 3 (bukan 2)
- Class weight: NEUTRAL × 2.0 (kelas minoritas + terlemah)
- max_len: 160 (sedikit lebih longgar dari 128)
- lr scheduler: cosine + warmup

**Estimasi waktu di T4:** ~60–90 menit (full data). Kuota Colab gratis cukup untuk 1 run.

**Output:** download `model_v2.zip` + copy metrics validation yang keluar.

In [ ]:
# 1. Setup
!git clone https://github.com/kusjay-space/sentiment-analysis.git
%cd sentiment-analysis
!pip install -q transformers==5.20.2 datasets==5.1.0 accelerate scikit-learn==1.9.1
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# PENTING: mount Google Drive supaya hasil selamat walau sesi mati / PC dimatikan.
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/sentiment-v2'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive siap:', DRIVE_DIR)

In [ ]:
# 2. Load full dataset + mapping
import random, numpy as np, torch
from datasets import load_dataset
from transformers import set_seed

SEED = 42
set_seed(SEED)

ID2LABEL = {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

def map_label(l):
    return 0 if l in (0,1) else (1 if l==2 else 2)

ds = load_dataset('SetFit/amazon_reviews_multi_en')
train_full = ds['train'].map(lambda b: {'sentiment':[map_label(l) for l in b['label']]}, batched=True)
val_full   = ds['validation'].map(lambda b: {'sentiment':[map_label(l) for l in b['label']]}, batched=True)

from collections import Counter
print('Train mapped dist:', Counter(train_full['sentiment']))
print('(0=NEG, 1=NEU, 2=POS)')

In [ ]:
# 3. Class weight — NEUTRAL dinaikkan 2x (minoritas + terlemah)
import numpy as np
counts = np.bincount(train_full['sentiment'])  # [80000, 40000, 80000]
n = counts.sum()
# base: n/(k*count) => NEG 0.83, NEU 1.67, POS 0.83; NEU tambahan boost
base = n / (3 * counts)
boost = np.array([1.0, 2.0, 1.0])  # NEUTRAL x2
class_weights = torch.tensor(base * boost, dtype=torch.float)
print('class_weights [NEG, NEU, POS]:', class_weights.tolist())

In [ ]:
# 4. Tokenisasi (max_len 160)
from transformers import AutoTokenizer
MAX_LEN = 160
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tok(b):
    return tokenizer(b['text'], truncation=True, max_length=MAX_LEN, padding='max_length')

keep = lambda ds: [c for c in ds.column_names if c != 'sentiment']
train_tok = train_full.map(tok, batched=True, remove_columns=keep(train_full)).rename_column('sentiment','labels')
val_tok   = val_full.map(tok, batched=True, remove_columns=keep(val_full)).rename_column('sentiment','labels')
train_tok.set_format('torch'); val_tok.set_format('torch')
print('Tokenized:', len(train_tok), 'train /', len(val_tok), 'val')

In [ ]:
# 5. Model + Weighted Trainer
from transformers import (AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score
import torch.nn as nn

model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        cw = class_weights.to(logits.device)
        loss = nn.functional.cross_entropy(logits, labels, weight=cw)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {'accuracy': accuracy_score(p.label_ids, preds),
            'f1_macro': f1_score(p.label_ids, preds, average='macro')}

fp16 = torch.cuda.is_available()
args = TrainingArguments(
    output_dir=DRIVE_DIR + '/checkpoint_v2',   # simpan ke Drive, bukan /content
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    fp16=fp16,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    seed=SEED,
    data_seed=SEED,
    report_to=[],
)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_tok,
                          eval_dataset=val_tok, processing_class=tokenizer,
                          data_collator=DataCollatorWithPadding(tokenizer),
                          compute_metrics=compute_metrics)

print(f'Mulai training: full 200k, 3 epoch, fp16={fp16}')
# resume otomatis kalau sesi sebelumnya sempat mati di tengah jalan
import glob
has_ckpt = len(glob.glob(DRIVE_DIR + '/checkpoint_v2/checkpoint-*')) > 0
print('Resume dari checkpoint:', has_ckpt)
trainer.train(resume_from_checkpoint=has_ckpt)
print('Selesai')

In [ ]:
# 6. Evaluasi validation + F1 per kelas
val_metrics = trainer.evaluate()
print('=== Validation Metrics (v2) ===')
for k,v in val_metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v,float) else f'  {k}: {v}')

# F1 per kelas
preds = trainer.predict(val_tok)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids
from sklearn.metrics import f1_score
f1s = f1_score(y_true, y_pred, average=None, labels=[0,1,2])
print('\nF1 per kelas:')
for i,name in ID2LABEL.items():
    print(f'  {name}: {f1s[i]:.4f}')
print('\nBandingkan v1: NEUTRAL 0.403, target v2 NEUTRAL > 0.50')

In [ ]:
# 7. Simpan ke Drive + download
model.save_pretrained('model/final_v2')
tokenizer.save_pretrained('model/final_v2')
!cd model && zip -r ../model_v2.zip final_v2/

# simpan juga ke Drive — ini yang bikin aman walau PC dimatikan
import shutil
shutil.copy('model_v2.zip', DRIVE_DIR + '/model_v2.zip')
print('Tersimpan di Drive:', DRIVE_DIR + '/model_v2.zip')

from google.colab import files
files.download('model_v2.zip')
print('\n=== LANGKAH SELANJUTNYA ===')
print('JIKA val accuracy v2 > v1 (0.75):')
print('1. Extract model_v2.zip -> ganti model/final/ di repo lokal')
print('2. Jalankan: python -m src.inference  (evaluasi test SEKALI)')
print('3. Paste metrics validation + test ke chat')
print('JIKA v2 <= v1: keep v1 (model/final tetap), laporkan saja')